In [1]:
!pip install transformers torch accelerate

In [2]:
import os

print(os.listdir("/kaggle/input/datasets/sujitham379/phase1-bengali-phi4"))

['phase1_results (2).csv']


In [4]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

# ======================
# CONFIG
# ======================

DATASET_PATH = "/kaggle/input/datasets/sujitham379/phase1-phi4-hin/phase1_results_hin_phi4.csv"
MODEL_NAME = "microsoft/phi-3-mini-4k-instruct"  # change for phi, llama etc
OUTPUT_FILE = "phase2_mechanistic_phi4_hin.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ======================

print("📂 Loading dataset...")
df = pd.read_csv(DATASET_PATH)
print("Columns:", df.columns)

# ======================
# LOAD MODEL (IMPORTANT FIX)
# ======================

print("🤖 Loading model with eager attention...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager"   # 🔥 required for attention
)

model = model.to(DEVICE)
model.eval()

# ======================
# FUNCTIONS
# ======================

def logit_lens(input_ids):
    with torch.no_grad():
        outputs = model(
            input_ids,
            output_hidden_states=True,
            return_dict=True
        )

    hidden_states = outputs.hidden_states
    lm_head = model.lm_head

    conf = []

    for layer_hidden in hidden_states:
        logits = lm_head(layer_hidden)
        probs = F.softmax(logits[:, -1, :], dim=-1)
        conf.append(torch.max(probs).item())

    return conf


def attention_metrics(input_ids):
    with torch.no_grad():
        outputs = model(
            input_ids,
            output_attentions=True,
            return_dict=True
        )

    if outputs.attentions is None:
        return np.nan, np.nan

    entropies = []
    self_ratios = []

    for layer_attn in outputs.attentions:
        attn = layer_attn[0]              # batch=1
        avg_attn = attn.mean(dim=0)       # avg heads

        # entropy
        entropy = -torch.sum(avg_attn * torch.log(avg_attn + 1e-9)).item()
        entropies.append(entropy)

        # self attention ratio (last token → itself)
        self_ratio = avg_attn[-1, -1].item()
        self_ratios.append(self_ratio)

    return np.mean(entropies), np.mean(self_ratios)


# ======================
# MAIN LOOP
# ======================

results = []

print("🔬 Running mechanistic evaluation...")

for _, row in tqdm(df.iterrows(), total=len(df)):

    # Use English question for stability
    q = str(row.get("question_en", ""))

    if len(q.strip()) == 0:
        continue

    inputs = tokenizer(
        q,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(DEVICE)

    # Logit lens
    layer_conf = logit_lens(inputs["input_ids"])

    early_conf = np.mean(layer_conf[:5])
    middle_conf = np.mean(layer_conf[len(layer_conf)//3:2*len(layer_conf)//3])
    late_conf = np.mean(layer_conf[-5:])

    # Attention
    entropy, self_ratio = attention_metrics(inputs["input_ids"])

    # Keep original row
    result_row = row.to_dict()

    result_row.update({
        "early_layer_confidence": early_conf,
        "middle_layer_confidence": middle_conf,
        "late_layer_confidence": late_conf,
        "attention_entropy": entropy,
        "self_attention_ratio": self_ratio
    })

    results.append(result_row)

# ======================
# SAVE
# ======================

pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)

print("✅ Phase 2 complete →", OUTPUT_FILE)

📂 Loading dataset...
Columns: Index(['id', 'model', 'language', 'question_en', 'question_hi', 'reference_en',
       'answer_generated_hi', 'generated_translated_en',
       'similarity_multilingual', 'similarity_translated', 'hallucinated',
       'drift_score', 'repetition_score', 'length_ratio'],
      dtype='object')
🤖 Loading model with eager attention...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

🔬 Running mechanistic evaluation...



100%|██████████| 817/817 [08:53<00:00,  1.53it/s]

✅ Phase 2 complete → phase2_mechanistic_phi4_hin.csv
